In [3]:

import numpy as np
import nltk
from nltk.corpus import stopwords
from tensorflow.keras.preprocessing import text
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, Lambda, Dense
import tensorflow.keras.backend as K
from sklearn.metrics.pairwise import cosine_similarity

# 1. Download stopwords
nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

# 2. Sample text dataset
data = [
    "Climate change is affecting ecosystems and communities around the world.",
    "Renewable energy sources such as solar and wind power help reduce carbon emissions.",
    "Forests play an important role in absorbing carbon dioxide from the atmosphere.",
    "Ocean pollution is a major threat to marine life and biodiversity.",
    "Recycling helps reduce waste and conserve valuable natural resources.",
    "Sustainable agriculture can improve food production while protecting the environment.",
    "Electric vehicles can help reduce pollution caused by traditional transportation.",
    "Governments and organizations are investing in clean energy technologies.",
    "Water conservation is important for maintaining healthy ecosystems and communities.",
    "Reducing plastic consumption can significantly decrease environmental pollution.",
    "Green buildings use energy efficient technologies to reduce their environmental impact.",
    "Wildlife conservation protects endangered species and their natural habitats.",
    "Planting trees helps improve air quality and supports biodiversity.",
    "Public transportation can reduce traffic congestion and carbon emissions.",
    "Environmental education encourages people to adopt sustainable lifestyles.",
    "Protecting natural resources is essential for future generations."
]

# 3. Tokenization and vocabulary
tokenizer = text.Tokenizer()

tokenizer.fit_on_texts([
    [w for w in text.text_to_word_sequence(doc) if w not in stop_words]
    for doc in data
])

word2id = tokenizer.word_index
word2id['PAD'] = 0
id2word = {v: k for k, v in word2id.items()}

wids = [
    [word2id[w] for w in text.text_to_word_sequence(doc) if w not in stop_words]
    for doc in data
]

vocab_size = len(word2id)
window_size = 2
embed_size = 100


# 4. Generate context-target pairs
X, Y = [], []

for words in wids:
    for i, word in enumerate(words):

        context = words[max(0, i-2):i] + words[i+1:i+3]

        context = pad_sequences(
            [context], maxlen=4
        )[0]

        X.append(context)
        Y.append(word)

X = np.array(X)
Y = to_categorical(Y, vocab_size)

print("Training Shape:", X.shape, Y.shape)


# 5. Create CBOW model
cbow = Sequential([
    Input(shape=(4,)),
    Embedding(vocab_size, embed_size),
    Lambda(lambda x: K.mean(x, axis=1)),
    Dense(vocab_size, activation='softmax')
])

# 6. Compile model
cbow.compile(
    optimizer='adam',
    loss='categorical_crossentropy'
)

# 7. Train model
cbow.fit(X, Y, epochs=50, batch_size=32)

# Extract word embeddings
weights = cbow.layers[0].get_weights()[0]

# 9. Find similar words
similarity = cosine_similarity(weights)

user_word = input("Enter a word: ").lower()

if user_word in word2id:
    idx = word2id[user_word]
    similar_idx = similarity[idx].argsort()[::-1]

    similar_words = [
        id2word[i] for i in similar_idx
        if i != idx and i != 0
    ][:5]

    print("Similar words:", similar_words)
else:
    print("Word not found in vocabulary")


Training Shape: (119, 4) (119, 92)
Epoch 1/50


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 4.5235 
Epoch 2/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 4.5103 
Epoch 3/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 4.4993 
Epoch 4/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 4.4885 
Epoch 5/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 4.4776 
Epoch 6/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 4.4666 
Epoch 7/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 4.4551 
Epoch 8/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 4.4435 
Epoch 9/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 4.4314 
Epoch 10/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 4.4186 
Epoch 11/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 4.4054 
Epoch 12/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 4.3917 
Epoch 13/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 4.3771 
Epoch 14/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 4.3617 
Epoch 15/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 4.3458 
Epoch 16/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/